In [2]:
import pandas as pd
import numpy as np
from scipy import stats

# 忽略jupyter无关警告，让输出更整洁
import warnings

from scipy.linalg import bandwidth

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print("库导入完成")

库导入完成


In [3]:
df_stock = pd.read_csv('stock_monthly.csv', encoding='utf-8-sig')
df_equity = pd.read_csv('equity_yearly.csv', encoding='utf-8-sig')
df_hs300 = pd.read_csv('hs300_monthly.csv', encoding='utf-8-sig')
df_rf = pd.read_csv('rf_daily.csv', encoding='utf-8-sig')

print(f"个股数据：{df_stock.shape[0]}行 × {df_stock.shape[1]}列")
print(f"股东权益：{df_equity.shape[0]}行 × {df_equity.shape[1]}列")
print(f"沪深300：{df_hs300.shape[0]}行 × {df_hs300.shape[1]}列")
print(f"无风险收益：{df_rf.shape[0]}行 × {df_rf.shape[1]}列")

个股数据：516572行 × 4列
股东权益：369040行 × 5列
沪深300：120行 × 3列
无风险收益：3653行 × 3列


In [4]:

print ("个股月度交易数据")
display (df_stock.head (3))
print ("年度股东权益数据")
display (df_equity.head (3))
print ("沪深 300 月收益率")
display (df_hs300.head (3))
print ("无风险收益率")
display (df_rf.head (3))
print ("-" * 80)
print ("数据表列名：")
print ("-" * 80)
print ("df_stock 列名：", list (df_stock.columns))
print ("df_equity 列名：", list (df_equity.columns))
print ("df_hs300 列名：", list (df_hs300.columns))
print ("df_rf 列名：", list (df_rf.columns))
print ("-" * 80)

个股月度交易数据


,Stkcd,Trdmnt,Msmvttl,Mretwd
0,1,2016-01,143086761.3900,-0.1660
1,1,2016-02,136790943.8900,-0.0440
2,1,2016-03,152244314.1200,0.1130


年度股东权益数据


,Stkcd,ShortName,Accper,Typrep,A003100000
0,1,平安银行,2015-12-31,A,161500000000.0000
1,1,平安银行,2016-01-01,A,161500000000.0000
2,1,平安银行,2016-03-31,A,187569000000.0000


沪深 300 月收益率


,Indexcd,Month,Idxrtn
0,300,2016-01,-0.2104
1,300,2016-02,-0.0233
2,300,2016-03,0.1184


无风险收益率


,Nrr1,Clsdt,Nrrmtdt
0,NRI01,2016-01-01,0.1241
1,NRI01,2016-01-02,0.1241
2,NRI01,2016-01-03,0.1241


--------------------------------------------------------------------------------
数据表列名：
--------------------------------------------------------------------------------
df_stock 列名： ['Stkcd', 'Trdmnt', 'Msmvttl', 'Mretwd']
df_equity 列名： ['Stkcd', 'ShortName', 'Accper', 'Typrep', 'A003100000']
df_hs300 列名： ['Indexcd', 'Month', 'Idxrtn']
df_rf 列名： ['Nrr1', 'Clsdt', 'Nrrmtdt']
--------------------------------------------------------------------------------


In [5]:
df_rf_clean = df_rf.copy ()
df_rf_clean.rename (columns={'Nrrmtdt': 'rf'}, inplace=True)
df_rf_clean ['month'] = pd.to_datetime (df_rf_clean ['Clsdt']).dt.to_period ('M')


df_rf_clean = df_rf_clean.drop_duplicates (subset=['month'], keep='first')
df_rf_clean = df_rf_clean.dropna (subset=['rf'])
df_rf_clean = df_rf_clean [df_rf_clean ['rf'] >= 0]
df_rf_clean = df_rf_clean [['month', 'rf']].reset_index (drop=True)

display (df_rf_clean.head ())

,month,rf
0,2016-01,0.1241
1,2016-02,0.1241
2,2016-03,0.1241
3,2016-04,0.1241
4,2016-05,0.1241


In [6]:
df_equity['Accper']=pd.to_datetime(df_equity['Accper'],errors='coerce')
df_equity = df_equity[(df_equity['Accper'].dt.month == 12) & (df_equity['Typrep'] == 'A')].copy()
df_equity['year'] = df_equity['Accper'].dt.year+1
df_equity = df_equity.drop_duplicates(subset=['Stkcd', 'year'], keep='first')
df_equity = df_equity.rename(columns={
    'Stkcd': 'code',
    'Accper': 'year_end_date',
    'A003100000': 'equity'
})

df_equity.head()

,code,ShortName,year_end_date,Typrep,equity,year
0,1,平安银行,2015-12-31,A,161500000000.0000,2016
5,1,平安银行,2016-12-31,A,202171000000.0000,2017
10,1,平安银行,2017-12-31,A,222054000000.0000,2018
15,1,平安银行,2018-12-31,A,240042000000.0000,2019
20,1,平安银行,2019-12-31,A,312983000000.0000,2020


In [7]:
df_stock_clean = df_stock.copy()
df_stock_clean['month']=pd.to_datetime(df_stock_clean['Trdmnt']).dt.to_period('M')
df_stock_clean = df_stock_clean.rename(columns={
    'Stkcd': 'code',
    'Mretwd': 'ret',
    'Msmvos': 'mktcap' ,
    'Trdmnt': 'year_end_month'
})
df_stock_clean = df_stock_clean.dropna(subset=['ret'])
df_stock_clean.head()

,code,year_end_month,Msmvttl,ret,month
0,1,2016-01,143086761.3900,-0.1660,2016-01
1,1,2016-02,136790943.8900,-0.0440,2016-02
2,1,2016-03,152244314.1200,0.1130,2016-03
3,1,2016-04,151242706.7900,-0.0066,2016-04
4,1,2016-05,150956533.2700,-0.0019,2016-05


In [8]:
df_stock_with_rf=pd.merge(left=df_stock_clean, right=df_rf_clean,on='month',how='left')
df_stock_with_rf=df_stock_with_rf.dropna(subset=['rf'])
df_stock_with_rf.head()

,code,year_end_month,Msmvttl,ret,month,rf
0,1,2016-01,143086761.3900,-0.1660,2016-01,0.1241
1,1,2016-02,136790943.8900,-0.0440,2016-02,0.1241
2,1,2016-03,152244314.1200,0.1130,2016-03,0.1241
3,1,2016-04,151242706.7900,-0.0066,2016-04,0.1241
4,1,2016-05,150956533.2700,-0.0019,2016-05,0.1241


In [9]:
df_stock_with_rf['eret']=df_stock_with_rf['ret']-df_stock_with_rf['rf']/100
df_stock_with_rf.head()

,code,year_end_month,Msmvttl,ret,month,rf,eret
0,1,2016-01,143086761.3900,-0.1660,2016-01,0.1241,-0.1672
1,1,2016-02,136790943.8900,-0.0440,2016-02,0.1241,-0.0452
2,1,2016-03,152244314.1200,0.1130,2016-03,0.1241,0.1117
3,1,2016-04,151242706.7900,-0.0066,2016-04,0.1241,-0.0078
4,1,2016-05,150956533.2700,-0.0019,2016-05,0.1241,-0.0031


In [10]:
df_factor = df_stock_with_rf[['month']].drop_duplicates().sort_values('month').reset_index(drop=True)

display(df_factor.head())

,month
0,2016-01
1,2016-02
2,2016-03
3,2016-04
4,2016-05


In [11]:
def cap_weighted_mean(group):
    # 加权平均计算函数
    return (group['eret'] * group['Msmvttl']).sum() / group['Msmvttl'].sum()

# 按月分组
df_mkt = df_stock_with_rf.groupby('month').apply(cap_weighted_mean).reset_index(name='Mkt_RF')

display(df_mkt.head())

,month,Mkt_RF
0,2016-01,-0.2393
1,2016-02,-0.0170
2,2016-03,0.1568
3,2016-04,-0.0137
4,2016-05,0.0016


In [12]:
df_factor = pd.merge(df_factor, df_mkt, on='month', how='inner')

display(df_factor.head())

,month,Mkt_RF
0,2016-01,-0.2393
1,2016-02,-0.0170
2,2016-03,0.1568
3,2016-04,-0.0137
4,2016-05,0.0016


In [13]:
monthly_median = df_stock_with_rf.groupby('month')['Msmvttl'].median().reset_index(name='median_mktcap')

df_stock_with_rf = pd.merge(df_stock_with_rf, monthly_median, on='month', how='left')

df_stock_with_rf['size_group'] = np.where(
    df_stock_with_rf['Msmvttl'] <= df_stock_with_rf['median_mktcap'],
    'S',
    'B'
)
df_stock_with_rf.head()

,code,year_end_month,Msmvttl,ret,month,rf,eret,median_mktcap,size_group
0,1,2016-01,143086761.3900,-0.1660,2016-01,0.1241,-0.1672,6382144.2300,B
1,1,2016-02,136790943.8900,-0.0440,2016-02,0.1241,-0.0452,6173085.0000,B
2,1,2016-03,152244314.1200,0.1130,2016-03,0.1241,0.1117,7570600.4000,B
3,1,2016-04,151242706.7900,-0.0066,2016-04,0.1241,-0.0078,7516982.8700,B
4,1,2016-05,150956533.2700,-0.0019,2016-05,0.1241,-0.0031,7543800.0000,B


In [14]:
group_eret = df_stock_with_rf.groupby(['month', 'size_group']).apply(cap_weighted_mean).reset_index(name='SMB_eret')
group_eret_pivot = group_eret.pivot(index='month', columns='size_group', values='SMB_eret').reset_index()


group_eret_pivot.columns = ['month', 'B', 'S']
group_eret_pivot['SMB'] = group_eret_pivot['S'] - group_eret_pivot['B']
df_smb = group_eret_pivot[['month', 'SMB']]


df_smb.head()

,month,SMB
0,2016-01,-0.0665
1,2016-02,-0.0094
2,2016-03,0.0508
3,2016-04,0.0258
4,2016-05,-0.0278


In [15]:

df_factor = pd.merge(df_factor, df_smb, on='month', how='inner')
df_factor.head()


,month,Mkt_RF,SMB
0,2016-01,-0.2393,-0.0665
1,2016-02,-0.0170,-0.0094
2,2016-03,0.1568,0.0508
3,2016-04,-0.0137,0.0258
4,2016-05,0.0016,-0.0278


In [16]:
df_stock_with_rf['year']=df_stock_with_rf['month'].dt.year
df_stock_with_rf.head()

,code,year_end_month,Msmvttl,ret,month,rf,eret,median_mktcap,size_group,year
0,1,2016-01,143086761.3900,-0.1660,2016-01,0.1241,-0.1672,6382144.2300,B,2016
1,1,2016-02,136790943.8900,-0.0440,2016-02,0.1241,-0.0452,6173085.0000,B,2016
2,1,2016-03,152244314.1200,0.1130,2016-03,0.1241,0.1117,7570600.4000,B,2016
3,1,2016-04,151242706.7900,-0.0066,2016-04,0.1241,-0.0078,7516982.8700,B,2016
4,1,2016-05,150956533.2700,-0.0019,2016-05,0.1241,-0.0031,7543800.0000,B,2016


In [17]:
df_stock_with_rf=pd.merge(df_stock_with_rf,
                          df_equity[['code','year','equity']],
                          on=['code','year'],
                          how='inner')
df_stock_with_rf.head()

,code,year_end_month,Msmvttl,ret,month,rf,eret,median_mktcap,size_group,year,equity
0,1,2016-01,143086761.3900,-0.1660,2016-01,0.1241,-0.1672,6382144.2300,B,2016,161500000000.0000
1,1,2016-02,136790943.8900,-0.0440,2016-02,0.1241,-0.0452,6173085.0000,B,2016,161500000000.0000
2,1,2016-03,152244314.1200,0.1130,2016-03,0.1241,0.1117,7570600.4000,B,2016,161500000000.0000
3,1,2016-04,151242706.7900,-0.0066,2016-04,0.1241,-0.0078,7516982.8700,B,2016,161500000000.0000
4,1,2016-05,150956533.2700,-0.0019,2016-05,0.1241,-0.0031,7543800.0000,B,2016,161500000000.0000


In [18]:
df_stock_with_rf['BM'] = df_stock_with_rf['equity'] / df_stock_with_rf['Msmvttl']

df_stock_with_rf = df_stock_with_rf.dropna(subset=['BM'])
df_stock_with_rf = df_stock_with_rf[df_stock_with_rf['BM'] > 0]

monthly_bm_quantile = df_stock_with_rf.groupby('month')['BM'].quantile([0.3, 0.7]).reset_index()
monthly_bm_quantile = monthly_bm_quantile.pivot(index='month', columns='level_1', values='BM').reset_index()
monthly_bm_quantile.columns = ['month', 'BM_30', 'BM_70']

df_stock_with_rf = pd.merge(df_stock_with_rf, monthly_bm_quantile, on='month', how='left')

df_stock_with_rf['bm_group'] = np.where(
    df_stock_with_rf['BM'] <= df_stock_with_rf['BM_30'],
    'L',
    np.where(
        df_stock_with_rf['BM'] >= df_stock_with_rf['BM_70'],
        'H',
        'M'
    )
)
bm_group_eret = df_stock_with_rf[df_stock_with_rf['bm_group'].isin(['L','H'])].groupby(['month','bm_group']).apply(cap_weighted_mean).reset_index(name='HML_eret')

bm_group_eret_pivot = bm_group_eret.pivot(index='month', columns='bm_group', values='HML_eret').reset_index()

bm_group_eret_pivot.columns = ['month','L','H']

bm_group_eret_pivot['HML'] = bm_group_eret_pivot['H'] - bm_group_eret_pivot['L']

df_hml = bm_group_eret_pivot[['month','HML']]
df_factor = pd.merge(df_factor, df_hml, on='month', how='inner')

display(df_factor.head(15))

,month,Mkt_RF,SMB,HML
0,2016-01,-0.2393,-0.0665,-0.0508
1,2016-02,-0.0170,-0.0094,0.0112
2,2016-03,0.1568,0.0508,0.0948
3,2016-04,-0.0137,0.0258,0.0095
4,2016-05,0.0016,-0.0278,0.0248
5,2016-06,0.0405,0.0395,0.0846
6,2016-07,0.0172,-0.0358,-0.0368
7,2016-08,0.0486,0.0155,0.0002
8,2016-09,-0.0173,0.0289,0.0126
9,2016-10,0.0363,0.0271,0.0115


In [19]:
factor_stats=df_factor[['Mkt_RF','SMB','HML']].describe()
display(factor_stats)

,Mkt_RF,SMB,HML
count,120.0000,120.0000,120.0000
mean,0.0135,-0.0152,0.0259
std,0.0554,0.0444,0.0581
min,-0.2393,-0.1489,-0.1388
25%,-0.0147,-0.0414,-0.0149
50%,0.0117,-0.0143,0.0219
75%,0.0374,0.0119,0.0684
max,0.2154,0.0855,0.1964


In [20]:
factor_corr=df_factor[['Mkt_RF','SMB','HML']].corr()
display(factor_corr)

,Mkt_RF,SMB,HML
Mkt_RF,1.0000,0.1520,0.4845
SMB,0.1520,1.0000,0.0896
HML,0.4845,0.0896,1.0000


In [21]:
for col in ['Mkt_RF','SMB','HML']:
    t_stat, p_value = stats.ttest_1samp (df_factor [col], 0)
    print (f"{col} 平均收益 t 统计量: {t_stat:.2f}, p 值: {p_value:.4f}")


Mkt_RF 平均收益 t 统计量: 2.67, p 值: 0.0088
SMB 平均收益 t 统计量: -3.75, p 值: 0.0003
HML 平均收益 t 统计量: 4.89, p 值: 0.0000


In [22]:
portfolios = df_stock_with_rf.groupby(['month', 'size_group', 'bm_group']).apply(
cap_weighted_mean
).reset_index(name='portfolio_eret')

portfolios['portfolio']=portfolios['size_group']+'_'+portfolios['bm_group']

portfolio_eret=portfolios.pivot(index='month',
                                columns='portfolio',
                                values='portfolio_eret').reset_index()
print (portfolio_eret.head(15))

portfolio    month     B_H     B_L     B_M     S_H     S_L     S_M
0          2016-01 -0.2007 -0.2573 -0.2659 -0.2998 -0.2645 -0.3166
1          2016-02 -0.0192 -0.0076 -0.0140 -0.0261 -0.0121 -0.0341
2          2016-03  0.1133  0.2077  0.1772  0.1610  0.2270  0.2039
3          2016-04 -0.0240 -0.0226 -0.0109 -0.0091  0.0167  0.0032
4          2016-05 -0.0038  0.0247  0.0107 -0.0508 -0.0120 -0.0294
5          2016-06 -0.0056  0.0815  0.0517  0.0322  0.0856  0.0685
6          2016-07  0.0275 -0.0062  0.0294  0.0042 -0.0293 -0.0223
7          2016-08  0.0474  0.0451  0.0346  0.0515  0.0578  0.0506
8          2016-09 -0.0228 -0.0137 -0.0200 -0.0020  0.0090 -0.0022
9          2016-10  0.0295  0.0358  0.0259  0.0301  0.0596  0.0354
10         2016-11  0.0669  0.0213  0.0250  0.0237  0.0415  0.0256
11         2016-12 -0.0464 -0.0463 -0.0459 -0.0355 -0.0516 -0.0517
12         2017-01  0.0307 -0.0166 -0.0133 -0.0188 -0.0238 -0.0487
13         2017-02  0.0206  0.0833  0.0482  0.0415  0.1136  0.

In [23]:
print("portfolio_eret 索引类型：", portfolio_eret.index.dtype)
print("df_factor 索引类型：", df_factor.index.dtype)


portfolio_eret 索引类型： int64
df_factor 索引类型： int64


In [24]:
import statsmodels.api as sm


reg_data = pd.concat([portfolio_eret, df_factor], axis=1).dropna()

for portfolio in portfolio_eret.columns:
    if portfolio == 'month':
        continue

    y = reg_data[portfolio]

    X = reg_data[['Mkt_RF', 'SMB', 'HML']]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': 6}
    )

    print(f"===== 组合 {portfolio} 回归结果 =====")
    print(model.summary())
    print("\n")


===== 组合 B_H 回归结果 =====
                            OLS Regression Results                            
Dep. Variable:                    B_H   R-squared:                       0.984
Model:                            OLS   Adj. R-squared:                  0.984
Method:                 Least Squares   F-statistic:                     1730.
Date:                Wed, 04 Feb 2026   Prob (F-statistic):           4.28e-96
Time:                        09:18:23   Log-Likelihood:                 444.49
No. Observations:                 120   AIC:                            -881.0
Df Residuals:                     116   BIC:                            -869.8
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0024      0

In [25]:
df_factor.to_csv('02_三因子结果表.csv', index=False, encoding='utf-8-sig')
print("三因子结果表 生成完成")

三因子结果表 生成完成
